In [9]:
import sys
import os

import pandas as pd
import csv
from transformers import AutoTokenizer, AutoModelForSequenceClassification, \
    Trainer, set_seed, TrainingArguments
from datasets import Dataset
import torch
from sklearn.utils import compute_class_weight
import numpy as np
from tqdm import tqdm
import json
import datasets
from sklearn.model_selection import train_test_split
from evaluate import load
import numpy as np

In [10]:
class Encoder():
    def __init__(self, model_identifier, dataset):
        self.model_id = model_identifier
        self.training_split = dataset["training_set"]
        self.validation_split = dataset["validation_set"]
        self.test_split = dataset["test_set"]
        self.dataset_name = dataset["dataset_name"] 
        self.dataset_language = dataset["dataset_language"]
        self.prediction_dir = f"./models/{model_identifier}_fine_tuned/predictions"
        self.output_dir = f"./models/{model_identifier}_fine_tuned"
        self.seed = 42
    
        self.tokenizer = AutoTokenizer.from_pretrained(model_identifier, trust_remote_code=True)
        print(f"Loaded tokenizer for model: {model_identifier}")
        self.model = AutoModelForSequenceClassification.from_pretrained(model_identifier, num_labels=2, trust_remote_code=True)
        print(f"Loaded model for sequence classification: {model_identifier} with 2 labels")


    def train(self):
        set_seed(self.seed)
        data = {
            "train" : self.__generate_data(self.training_split)[0], # tokenized dataset
            "val" : self.__generate_data(self.validation_split)[0] # tokenized dataset
        }   
        # compute class weight (in case labels are unbalanced)        
        try:
            #class_weights = compute_class_weight(
            #    "balanced",
            #    classes=np.unique(data["train"]["labels"].float().numpy()),
            #    y=data["train"]["labels"].float().numpy()).astype("float32")
            
            labels_array = np.array(data["train"]["labels"]).astype(int)  # converte la lista in array NumPy

            class_weights = compute_class_weight(
                "balanced",
                classes=np.unique(labels_array),
                y=labels_array
            ).astype("float32")
        except Exception as e:
            print("Unable to balance classes")
            class_weights = np.array([1, 1]).astype("float32")

        print("Class weights:", class_weights)


        print('We will use the device:', torch.cuda.get_device_name(0))
        training_args = TrainingArguments(
            seed=self.seed,
            output_dir=self.output_dir,
            num_train_epochs=5,
            learning_rate=2e-5,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            save_strategy="epoch",
            eval_strategy="epoch",
            logging_strategy="steps",
            logging_steps=50, # ogni quanti step loggare
            overwrite_output_dir=True,
            report_to=None,
            load_best_model_at_end=True,
            weight_decay=0.01,   # Standard per BERT/RoBERTa (CONSIGLIATO)
            warmup_ratio=0.1 # aumenta gradualmente il learning rate
        )


        trainer = CustomTrainer(
            model=self.model,
            args=training_args,
            train_dataset=data["train"],
            eval_dataset=data["val"],
            compute_metrics=self.__compute_metrics
        )
        trainer.set_class_weights(class_weights)
        trainer.train()
        return trainer
    

    def predict(self, trainer):
        test_data, ids = self.__generate_data(self.test_split)
        predictions = trainer.predict(test_data)

        if not os.path.exists(self.prediction_dir): 
            os.makedirs(self.prediction_dir)  
        with open(f"{self.prediction_dir}/predictions_{self.dataset_name}_{self.dataset_language}.csv", "w", newline='', encoding='utf-8') as fo:
            writer = csv.DictWriter(fo, fieldnames=["unique_id", "predicted_label", "label"])
            writer.writeheader()
            for i, pred in enumerate(predictions.predictions):
                predicted_label = int(np.argmax(pred))
                writer.writerow({
                    "unique_id": ids[i],
                    "predicted_label": predicted_label,
                    "label": int(predictions.label_ids[i])
                })


    def __generate_data(self, split):
        ids, texts, labels = [], [], []

        for _, row in split.iterrows():
            ids.append(row['unique_id'])
            texts.append(row['text'])
            labels.append(row['label'])

        df = pd.DataFrame({"text":texts, "labels":labels})
        dt = Dataset.from_pandas(df)
        tokenized_dataset = dt.map(lambda x: self.__tokenize(x, self.tokenizer), remove_columns=['text'])
        tokenized_dataset = tokenized_dataset.with_format("torch")
        return tokenized_dataset, ids
    
    @staticmethod
    def __compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        
        # Carica le metriche da Hugging Face Evaluate
        accuracy_metric = load("accuracy")
        precision_metric = load("precision")
        recall_metric = load("recall")
        f1_metric = load("f1")
        
        # Calcola tutte le metriche
        accuracy = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
        precision = precision_metric.compute(predictions=preds, references=labels, average="weighted")["precision"]
        recall = recall_metric.compute(predictions=preds, references=labels, average="weighted")["recall"]
        f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
        
        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }


    def __tokenize(self, x, tokenizer):
        return tokenizer(
            x["text"], 
            padding="max_length", #config.padding,
            truncation=True, #config.truncation,
            max_length=512, #config.max_length,
        )

class CustomTrainer(Trainer):
    """ Custom Trainer class to implement a custom loss function
    """

    def set_class_weights(self, class_weights):
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Focal Loss: https://arxiv.org/abs/1708.02002
        """
        gamma = 2.0
        alpha = 0.25
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Focal Loss Perseval
        #loss_fct = torch.nn.CrossEntropyLoss(weight=torch.tensor(self.class_weights)).to("cuda")
        #BCEloss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        #pt = torch.exp(-BCEloss)  # prevents nans when probability
        #loss = alpha * (1 - pt) ** gamma * BCEloss

        # Focal Loss (corretta?)
        loss_fct = torch.nn.CrossEntropyLoss(weight=torch.tensor(self.class_weights).to(logits.device), reduction='none')
        BCEloss = loss_fct(logits, labels)  # shape: [batch_size]
        pt = torch.exp(-BCEloss) # Probabilità corrette per la classe target
        loss = alpha * (1 - pt) ** gamma * BCEloss # Focal loss
        loss = loss.mean() # Media sul batch

        # Binary Cross-Entropy loss calculation (https://medium.com/data-scientists-diary/implementing-focal-loss-in-pytorch-for-class-imbalance-24d8aa3b59d9)
        #loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        #bce_loss = loss_fct(logits, labels)  # shape: [batch_size]
        #pt = torch.exp(-bce_loss)  # Convert BCE loss to probability -> Compute pt (model confidence on true class)
        #focal_loss = alpha * (1 - pt) ** gamma * bce_loss  # Apply the focal adjustment
        #loss = focal_loss.mean()

        return (loss, outputs) if return_outputs else loss


In [11]:
def load_dataset(file_path, language='en', dataset_name='MultiPICo', limit=None):
    """
    Carica il dataset dal unified corpus, mantenendo gli unique_id anche negli split.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            unified_corpus = json.load(f)
        print(f"Loaded unified corpus with {len(unified_corpus)} entries")
        
        df = pd.DataFrame(unified_corpus)
        
        # Filtra per lingua e dataset
        df_filtered = df[(df['language'] == language) & (df['dataset_name'] == dataset_name)]
        print(f"Filtered {dataset_name} {language.upper()} entries: {len(df_filtered)}")
        
        # Limita il numero di entries se necessario
        if limit is not None:
            df_filtered = df_filtered.head(limit)
            print(f"Using {len(df_filtered)} entries (limited)")

        # Seleziona solo le colonne utili
        df_filtered = df_filtered[['unique_id', 'post', 'annotations']].copy()
        df_filtered = df_filtered.rename(columns={'post': 'text'})
        df_filtered['label'] = df_filtered['annotations'].apply(
            lambda x: x.get('irony_appropriateness_label', None)
        )
        df_filtered = df_filtered.drop(columns=['annotations'])

        print(f"Entries with label 1: {len(df_filtered[df_filtered['label'] == 1])}")
        print(f"Entries with label 0: {len(df_filtered[df_filtered['label'] == 0])}")

        # Includiamo anche gli ID nello split
        train_df, temp_df = train_test_split(
            df_filtered,
            test_size=0.3,
            random_state=42,
            stratify=df_filtered['label']
        )

        val_df, test_df = train_test_split(
            temp_df,
            test_size=0.5,
            random_state=42,
            stratify=temp_df['label']
        )

        # Mostra distribuzioni
        for name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
            print(f'{name} len: {len(split_df)} | label 1: {len(split_df[split_df["label"]==1])} | label 0: {len(split_df[split_df["label"]==0])}')

        dataset_splits = {
            'dataset_name': dataset_name,
            'dataset_language': language,
            'training_set': train_df.reset_index(drop=True),
            'validation_set': val_df.reset_index(drop=True),
            'test_set': test_df.reset_index(drop=True)
        }

        return dataset_splits
        
    except Exception as e:
        print(f"Error loading unified corpus: {e}")
        raise e


In [12]:
data_path = "../1_datasets_merge/output/unified_corpus.json"

# Carica e filtra il dataset
dataset_splits = load_dataset(
    file_path=data_path,
    language='en',
    dataset_name='MultiPICo',
    limit=None  # None per usare tutti i dati, oppure un numero specifico per test
)

Loaded unified corpus with 25814 entries
Filtered MultiPICo EN entries: 2999
Entries with label 1: 882
Entries with label 0: 2117
train len: 2099 | label 1: 617 | label 0: 1482
val len: 450 | label 1: 133 | label 0: 317
test len: 450 | label 1: 132 | label 0: 318


In [ ]:
models = {
    #"eurobert": "EuroBERT/EuroBERT-210m",
    "xlm-roberta-base": "FacebookAI/xlm-roberta-base"
}

for model_name, model_identifier in models.items():
    print(f"Training and evaluating model: {model_name} ({model_identifier})")
    model = Encoder(model_identifier, dataset_splits)
    trainer = model.train()
    model.predict(trainer)

    del model
    torch.cuda.empty_cache()

Training and evaluating model: xlm-roberta-base (FacebookAI/xlm-roberta-base)
Loaded tokenizer for model: FacebookAI/xlm-roberta-base


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model for sequence classification: FacebookAI/xlm-roberta-base with 2 labels


Map: 100%|██████████| 450/450 [00:00<00:00, 1665.62 examples/s]


Class weights: [0.70816463 1.7009724 ]
We will use the device: NVIDIA GeForce RTX 4050 Laptop GPU


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.050000,0.047543,0.295556,0.087353,0.295556,0.134850


c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
import os 
import pandas as pd
from evaluate import load

In [ ]:
path = "./models/"
for model_folder in os.listdir(path):
    fine_tuned_path = os.path.join(path, model_folder)
    for subfolder in os.listdir(fine_tuned_path):
        subfolder_path = os.path.join(fine_tuned_path, subfolder, "predictions")
        print(f"Evaluating predictions in: {subfolder_path}")
        if os.path.exists(subfolder_path):
            for file in os.listdir(subfolder_path):
                if file.endswith(".csv"):
                    file_path = os.path.join(subfolder_path, file)
                    df = pd.read_csv(file_path)
                    accuracy_metric = load("accuracy")
                    precision_metric = load("precision")
                    recall_metric = load("recall")
                    f1_metric = load("f1")

                    accuracy = accuracy_metric.compute(predictions=df["predicted_label"], references=df["label"])["accuracy"]
                    precision = precision_metric.compute(predictions=df["predicted_label"], references=df["label"], average="weighted")["precision"]
                    recall = recall_metric.compute(predictions=df["predicted_label"], references=df["label"], average="weighted")["recall"]
                    f1 = f1_metric.compute(predictions=df["predicted_label"], references=df["label"], average="weighted")["f1"]

                    print(f"Metrics for {file_path}:")
                    print(f"  Accuracy: {accuracy:.4f}")
                    print(f"  Precision: {precision:.4f}")
                    print(f"  Recall: {recall:.4f}")
                    print(f"  F1 Score: {f1:.4f}")


Evaluating predictions in: ./models/EuroBERT\EuroBERT-210m_fine_tuned\predictions
Evaluating predictions in: ./models/EuroBERT\EuroBERT-210m_fine_tuned_2e-5_3_epoch\predictions
Metrics for ./models/EuroBERT\EuroBERT-210m_fine_tuned_2e-5_3_epoch\predictions\predictions_MultiPICo_en.csv:
  Accuracy: 0.5578
  Precision: 0.6771
  Recall: 0.5578
  F1 Score: 0.5750
Evaluating predictions in: ./models/FacebookAI\xlm-roberta-base_fine_tuned\predictions


c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Metrics for ./models/FacebookAI\xlm-roberta-base_fine_tuned\predictions\predictions_MultiPICo_en.csv:
  Accuracy: 0.7067
  Precision: 0.4994
  Recall: 0.7067
  F1 Score: 0.5852
Evaluating predictions in: ./models/FacebookAI\xlm-roberta-base_fine_tuned_2e-5_3_epoch\predictions
Metrics for ./models/FacebookAI\xlm-roberta-base_fine_tuned_2e-5_3_epoch\predictions\predictions_MultiPICo_en.csv:
  Accuracy: 0.5000
  Precision: 0.6598
  Recall: 0.5000
  F1 Score: 0.5106
Evaluating predictions in: ./models/FacebookAI\xlm-roberta-base_fine_tuned_2r-5_3_epoch_fl_perseval\predictions
Metrics for ./models/FacebookAI\xlm-roberta-base_fine_tuned_2r-5_3_epoch_fl_perseval\predictions\predictions_MultiPICo_en.csv:
  Accuracy: 0.6467
  Precision: 0.6665
  Recall: 0.6467
  F1 Score: 0.6547
Evaluating predictions in: ./models/FacebookAI\xlm-roberta-base_fine_tuned_5e-6_5_epco_fl_perseval\predictions
Metrics for ./models/FacebookAI\xlm-roberta-base_fine_tuned_5e-6_5_epco_fl_perseval\predictions\predictions_